# 05 — Outlier Handling

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 10.

Outliers corrupt forecasting models — a single anomalous spike can distort
seasonal indices and trend estimates for months. Three approaches covered here:

1. **Winsorization** — cap extreme values at a percentile threshold
2. **Standard deviation method** — flag values beyond ±k standard deviations
3. **Error standard deviation** — flag based on forecast error distribution

**Supply chain context:** Promotional spikes, stockouts, data entry errors,
COVID-era demand shocks — all create outliers that must be handled before modelling.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 1. Simulate Demand with Outliers

In [ ]:
np.random.seed(42)
demand = np.random.normal(1000, 100, 48)

# Inject outliers: promotional spike + data entry error + stockout
demand[12] = 2800   # promotional event spike
demand[25] = 4500   # data entry error
demand[38] = 100    # stockout / supply disruption

demand = np.clip(demand, 0, None)
print(f'Demand range: {demand.min():.0f} — {demand.max():.0f}')

## 2. Method 1 — Winsorization

In [ ]:
def winsorise(d, lower_pct=5, upper_pct=95):
    """Cap values below lower_pct and above upper_pct percentile."""
    lower = np.percentile(d, lower_pct)
    upper = np.percentile(d, upper_pct)
    return np.clip(d, lower, upper)

demand_winsorised = winsorise(demand)
print(f'After winsorisation — Max: {demand_winsorised.max():.0f}')

## 3. Method 2 — Standard Deviation Threshold

In [ ]:
def std_outlier_replace(d, k=2.0):
    """Replace values beyond k std devs with the mean."""
    d = d.copy()
    mean, std = d.mean(), d.std()
    upper = mean + k * std
    lower = mean - k * std
    outlier_mask = (d > upper) | (d < lower)
    d[outlier_mask] = mean
    print(f'Outliers detected: {outlier_mask.sum()} periods')
    return d, outlier_mask

demand_cleaned, mask = std_outlier_replace(demand, k=2.0)

## 4. Visualise All Three Versions

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(demand, color='#D85A30', linewidth=1.2)
axes[0].set_title('Raw Demand — with outliers', fontsize=11)
axes[0].set_ylabel('Units')
axes[0].grid(alpha=0.3)

axes[1].plot(demand_winsorised, color='#185FA5', linewidth=1.2)
axes[1].set_title('Winsorised (5th–95th percentile)', fontsize=11)
axes[1].set_ylabel('Units')
axes[1].grid(alpha=0.3)

axes[2].plot(demand_cleaned, color='#1D9E75', linewidth=1.2)
axes[2].scatter(np.where(mask)[0], demand_cleaned[mask],
                color='red', zorder=5, label='Replaced', s=60)
axes[2].set_title('Std Dev Method (±2σ replaced with mean)', fontsize=11)
axes[2].set_ylabel('Units')
axes[2].set_xlabel('Period (months)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Outlier Handling Methods Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Key Takeaways

- Always inspect and clean data **before** fitting any forecasting model
- Winsorization is non-destructive — it preserves direction but caps magnitude
- Std dev method works well when outliers are symmetric around the mean
- Document every outlier decision — supply chain stakeholders will ask why you changed a data point

**Next:** Forecast KPIs (`06_forecast_kpis.ipynb`) — how to measure model performance.
